# Notebook 1 — Predict **total latency (seconds)** per compression config

**LightGBM, 9-value output.** For a single prompt this model predicts `total_latency_sec` under each of the 9 compression configurations (KVQuant 2/3/4-bit, H2O 20/40/60%, RocketKV 8/16/32x) — a length-9 vector.

**Protocol.** (1) Patch the prompts (GSM8K fewshot prefix; ARC/HellaSwag question+choices), then reshape to one row per prompt with 9 target columns. (2) Hold out **15%** as a test set once, stratified by dataset. (3) On the remaining **85%**, tune hyperparameters with **repeated stratified k-fold CV** (5 folds × 3 repeats), selecting the config with the lowest mean CV MAE — the test set is never seen during tuning. (4) Refit on all 85% and evaluate the 15% test set once. Targets are fit in log1p space and inverted, so all errors are in seconds.

## Setup

In [ ]:
# Install deps (Colab-safe; no-op if already present). Add --break-system-packages
# locally if pip refuses to touch a managed environment.
!pip install -q lightgbm scikit-learn scipy pandas numpy joblib requests
import os, warnings
os.makedirs("artifacts", exist_ok=True)
warnings.filterwarnings("ignore", message="X does not have valid feature names")


## Engine — data loading, prompt patching, reshape, features

In [ ]:
# ===========================================================================
# ENGINE — shared across the latency / memory / correctness notebooks.
# Only the CONFIG block below changes between the three notebooks.
# ===========================================================================
import io, os, re, ast, time, itertools
import numpy as np, pandas as pd, requests

# ------------------------------- CONFIG ------------------------------------
TARGET   = "total_latency_sec"
ARTIFACT = "reg_total_latency_lightgbm.joblib"
# ---------------------------------------------------------------------------

RS = 42                    # global random seed — one split, reproducible everywhere
TEST_FRAC = 0.15           # 15% held out for the final test; 85% for train+val+CV

# The 9 compression configurations. THIS is the output axis: every model emits a
# length-9 vector, one predicted value per configuration, for a single prompt.
CONFIGS = ["kvquant_2bit", "kvquant_3bit", "kvquant_4bit",
           "h2o_20", "h2o_40", "h2o_60",
           "rocketkv_8x", "rocketkv_16x", "rocketkv_32x"]

# Map each config to the on-disk / on-repo filename stem.
FNAME = {"kvquant_2bit": "kvquant_2bit", "kvquant_3bit": "kvquant_3bit", "kvquant_4bit": "kvquant_4bit",
         "h2o_20": "h2o_budget_20pct", "h2o_40": "h2o_budget_40pct", "h2o_60": "h2o_budget_60pct",
         "rocketkv_8x": "rocketkv_ratio_8x", "rocketkv_16x": "rocketkv_ratio_16x", "rocketkv_32x": "rocketkv_ratio_32x"}

DATASETS = ["gsm8k", "arc_challenge", "hellaswag"]
IS_MCQ   = {"gsm8k": False, "arc_challenge": True, "hellaswag": True}

# Load CSVs from a local ./Data (if the repo is checked out) or from raw GitHub.
LOCAL_DIRS = ["Data", "../Data", "../../Data"]
REPO_RAW   = "https://raw.githubusercontent.com/yoshikodes/KVCacheCompression/main/Data"

def _local_dir():
    for d in LOCAL_DIRS:
        if os.path.isdir(d) and any(f.endswith("_per_prompt.csv") for f in os.listdir(d)):
            return d
    return None

_LOCAL = _local_dir()

def _read(cfg, ds):
    """Read one per-prompt CSV; standardize the index column name to 'idx'."""
    fn = f"{FNAME[cfg]}_{ds}_per_prompt.csv"
    if _LOCAL:
        df = pd.read_csv(os.path.join(_LOCAL, fn))
    else:
        r = requests.get(f"{REPO_RAW}/{fn}", timeout=60); r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
    idxc = "question_index" if "question_index" in df.columns else "item_index"
    return df.rename(columns={idxc: "idx"})

print("Data source:", _LOCAL if _LOCAL else REPO_RAW)

# ===========================================================================
# PROMPT PATCHING  (verbatim logic from the repo's add_prompt_context.py)
# ---------------------------------------------------------------------------
#   * GSM8K : the stored prompt is just the raw question. The real model input
#             was the 8-shot fewshot prefix + "\nQuestion: <q>\nAnswer:".
#   * ARC / HellaSwag : the stored prompt is the question/context only; the
#             options live in a separate `choices` column. We merge them into a
#             delimited "Answer choices:" block (labels taken from `choices`;
#             HellaSwag has no native label so A,B,C,... are assigned in order).
# Both transforms are IDEMPOTENT (guarded by the same markers the repo uses), so
# they do the right thing whether or not a given CSV was already patched.
# ===========================================================================
GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation step by step, then end with exactly this format:\n"
    "#### <final number>\n\n"
    "Question: There are 15 trees in the grove. Grove workers will plant trees today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n"
    "Answer: There are 15 trees originally. After planting, there are 21 trees. So the workers planted 21 - 15 = 6 trees.\n"
    "#### 6\n\n"
    "Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n"
    "Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5 cars.\n"
    "#### 5\n\n"
    "Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n"
    "Answer: Leah and her sister started with 32 + 42 = 74 chocolates. After eating 35, they have 74 - 35 = 39 left.\n"
    "#### 39\n\n"
    "Question: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\n"
    "Answer: Jason started with 20 lollipops and now has 12. So he gave away 20 - 12 = 8 lollipops.\n"
    "#### 8\n\n"
    "Question: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\n"
    "Answer: Shawn started with 5 toys. He got 2 from mom and 2 from dad, which is 2 + 2 = 4 more toys. 5 + 4 = 9 toys total.\n"
    "#### 9\n\n"
    "Question: There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?\n"
    "Answer: 4 days from Monday to Thursday, with 5 computers installed each day, is 4 * 5 = 20 computers added. 9 + 20 = 29 computers total.\n"
    "#### 29\n\n"
    "Question: Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?\n"
    "Answer: Michael started with 58 golf balls. After losing 23 on Tuesday, he had 58 - 23 = 35. After losing 2 more on Wednesday, he had 35 - 2 = 33 golf balls.\n"
    "#### 33\n\n"
    "Question: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\n"
    "Answer: Five bagels at $3 each cost 5 * 3 = 15 dollars. Olivia started with $23, so she has 23 - 15 = 8 dollars left.\n"
    "#### 8\n"
)
GSM8K_Q_LEAD = "\nQuestion: "
GSM8K_A_TAIL = "\nAnswer:"
CHOICES_BLOCK_HEADER = "\n\nAnswer choices:\n"

def _format_choices_block(choices):
    lines = []
    for i, entry in enumerate(choices):
        if isinstance(entry, (list, tuple)) and len(entry) == 2:
            label, text = entry
        else:
            label, text = chr(ord("A") + i), entry
        lines.append(f"{label}. {text}")
    return CHOICES_BLOCK_HEADER + "\n".join(lines)

def patch_gsm8k(prompt):
    p = str(prompt)
    if p.startswith(GSM8K_FEWSHOT_PREFIX):          # already patched -> leave it
        return p
    return GSM8K_FEWSHOT_PREFIX + GSM8K_Q_LEAD + p + GSM8K_A_TAIL

def patch_mc(prompt, choices_cell):
    p = str(prompt)
    if CHOICES_BLOCK_HEADER in p:                    # already patched -> leave it
        return p
    try:
        choices = ast.literal_eval(choices_cell)
    except (ValueError, SyntaxError):
        return p                                     # unparseable -> leave question as-is
    return p + _format_choices_block(choices)

# ===========================================================================
# BUILD THE WIDE TABLE: one row per (dataset, prompt); 9 target columns.
# The bare question is identical across all 9 configs for a given index (the
# eval order is seed-42 stable across methods), so we take the patched
# `model_input` from a single reference config and attach the 9 target values.
# ===========================================================================
def build_wide():
    frames = []
    for ds in DATASETS:
        ref = _read("kvquant_2bit", ds)
        cols = ["idx", "prompt"] + (["choices"] if IS_MCQ[ds] else [])
        ref = ref[cols]
        if IS_MCQ[ds]:
            model_input = [patch_mc(p, c) for p, c in zip(ref["prompt"], ref["choices"])]
        else:
            model_input = [patch_gsm8k(p) for p in ref["prompt"]]
        base = pd.DataFrame({"dataset": ds, "idx": ref["idx"].values, "model_input": model_input})
        for cfg in CONFIGS:                          # attach each config's target column
            d = _read(cfg, ds)[["idx", TARGET]].rename(columns={TARGET: cfg})
            base = base.merge(d, on="idx", how="inner")
        frames.append(base)
    wide = pd.concat(frames, ignore_index=True)
    assert int(wide[CONFIGS].isna().sum().sum()) == 0, "unexpected NaN targets"
    return wide

# ===========================================================================
# FEATURES: cheap numeric descriptions of the (patched) prompt text, plus a
# one-hot of the dataset. NOTE: no feature encodes the compression config —
# the config is the OUTPUT axis, so a single prompt maps to one feature row and
# nine predicted values.
# ===========================================================================
def _feats(p):
    s = str(p); t = s.split(); nt = max(len(t), 1); nc = max(len(s), 1)
    return {
        "n_char":      len(s),
        "n_tok":       len(t),
        "ttr":         len(set(t)) / nt,                                   # type-token ratio
        "digit_ratio": sum(c.isdigit() for c in s) / nc,
        "punct_ratio": sum(not c.isalnum() and not c.isspace() for c in s) / nc,
        "upper_ratio": sum(c.isupper() for c in s) / nc,
        "avg_tok":     nc / nt,                                            # mean word length
        "num_count":   len(re.findall(r"\d+", s)),
        "has_q":       int("?" in s),
        "n_newline":   s.count("\n"),
    }

DS_DUMMY_COLS = [f"ds_{d}" for d in DATASETS]        # fixed column order for the one-hot

def fmat(df):
    X = pd.DataFrame([_feats(p) for p in df["model_input"]])
    dummies = pd.get_dummies(df["dataset"], prefix="ds").reindex(columns=DS_DUMMY_COLS, fill_value=0)
    return pd.concat([X.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

# Target matrix (n_prompts, 9), column order == CONFIGS.
def Ymat(df):
    return df[CONFIGS].to_numpy(dtype=float)


## Load, patch prompts, and hold out the test set once

In [ ]:
from sklearn.model_selection import train_test_split

# Build the wide table (patches prompts, attaches the 9 targets).
wide = build_wide()
print("wide table:", wide.shape, "| rows per dataset:", wide["dataset"].value_counts().to_dict())

# --------------------------------------------------------------------------
# STEP 1 — hold out a TEST set ONCE, stratified by dataset so the 15% test has
# the same gsm8k/arc/hellaswag mix as the whole. The test set is NOT looked at
# during tuning; it is touched exactly once, at the very end.
# --------------------------------------------------------------------------
tr_parts, te_parts = [], []
for ds, g in wide.groupby("dataset"):
    a, b = train_test_split(g, test_size=TEST_FRAC, random_state=RS, shuffle=True)
    tr_parts.append(a); te_parts.append(b)

trainval_df = pd.concat(tr_parts).sample(frac=1, random_state=RS).reset_index(drop=True)
test_df     = pd.concat(te_parts).sample(frac=1, random_state=RS).reset_index(drop=True)

print("train+val:", len(trainval_df), "| test (held out):", len(test_df))
print("per-dataset train+val:", trainval_df["dataset"].value_counts().to_dict())

# Peek at one patched prompt per dataset so the patching is visible.
for ds in DATASETS:
    ex = wide[wide.dataset == ds]["model_input"].iloc[0]
    print(f"\n--- patched model_input | {ds} (first {180} chars) ---\n{ex[:180]!r}")


## Targets

In [ ]:
# --------------------------------------------------------------------------
# Targets are strictly positive and span a wide range (seconds / GB across very
# different datasets), so we fit in log1p space and invert with expm1 for all
# reported predictions and metrics — errors stay in the ORIGINAL units.
# --------------------------------------------------------------------------
from sklearn.metrics import mean_absolute_error, r2_score

def to_train_space(Y):   return np.log1p(Y)
def to_orig_space(Yhat): return np.expm1(Yhat)

# Quick sanity print of the 9-way target ranges.
_Y = Ymat(trainval_df)
print("target:", TARGET)
for j, c in enumerate(CONFIGS):
    print(f"  {c:14s} min={_Y[:,j].min():.4g}  med={np.median(_Y[:,j]):.4g}  max={_Y[:,j].max():.4g}")


## Hyperparameter search — repeated stratified k-fold CV (train+val only)

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import RepeatedStratifiedKFold

# ==========================================================================
# STEP 2 — choose hyperparameters with REPEATED k-fold CV on train+val ONLY.
# Repeated k-fold = run k-fold N_REPEATS times, each with a fresh random
# partition; averaging over all folds*repeats gives a stabler estimate and a
# real error bar. Folds are stratified by dataset to keep the regime mix.
# The test set is NOT touched here.
# ==========================================================================
NUM_LEAVES_GRID   = [15, 31, 63]
MIN_CHILD_GRID    = [20, 50, 100]
N_ESTIMATORS_GRID = [300, 600]
CANDIDATES = [{"num_leaves": nl, "min_child_samples": mc, "n_estimators": ne}
              for nl, mc, ne in itertools.product(NUM_LEAVES_GRID, MIN_CHILD_GRID, N_ESTIMATORS_GRID)]

FIXED = dict(learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
             random_state=RS, verbose=-1, n_jobs=-1)

N_FOLDS, N_REPEATS = 5, 3
print(f"{len(CANDIDATES)} candidates x {N_FOLDS}-fold x {N_REPEATS} repeats "
      f"x 9 outputs = {len(CANDIDATES)*N_FOLDS*N_REPEATS*9} model fits\n")

def fit_9(cfg, Xtr, Ytr_log):
    """Train one LGBMRegressor per config (9 total) in log space."""
    return [lgb.LGBMRegressor(**cfg, **FIXED).fit(Xtr, Ytr_log[:, j])
            for j in range(len(CONFIGS))]

def predict_9(models, X):
    return to_orig_space(np.column_stack([m.predict(X) for m in models]))

def cv_score(cfg):
    rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=RS)
    strat = trainval_df["dataset"].values
    maes, r2s = [], []
    for tr_idx, va_idx in rskf.split(trainval_df, strat):
        tr, va = trainval_df.iloc[tr_idx], trainval_df.iloc[va_idx]
        Xtr, Xva = fmat(tr).values, fmat(va).values
        models = fit_9(cfg, Xtr, to_train_space(Ymat(tr)))
        pred = predict_9(models, Xva)          # original units
        yt = Ymat(va)
        # mean-across-9-outputs MAE / R2 for this fold
        maes.append(np.mean([mean_absolute_error(yt[:, j], pred[:, j]) for j in range(9)]))
        r2s.append(np.mean([r2_score(yt[:, j], pred[:, j]) for j in range(9)]))
    return np.array(maes), np.array(r2s)

results = []
for i, cfg in enumerate(CANDIDATES, 1):
    mae, r2 = cv_score(cfg)
    results.append((cfg, mae.mean(), mae.std(), r2.mean()))
    print(f"[{i:>2}/{len(CANDIDATES)}] {str(cfg):<58} CV MAE={mae.mean():.4f}±{mae.std():.4f}  R2={r2.mean():.4f}")

# Select the config with the LOWEST mean CV MAE (chosen on train+val only).
best_cfg = min(results, key=lambda r: r[1])[0]
print("\nTop 5 by CV MAE (lower is better):")
for cfg, m, s, r2 in sorted(results, key=lambda r: r[1])[:5]:
    print(f"  {str(cfg):<58} MAE={m:.4f}±{s:.4f}  R2={r2:.4f}")
print("\nBEST config (test set never used):", best_cfg)


## Final evaluation — test set touched once

In [ ]:
# ==========================================================================
# STEP 3 — final fit on ALL train+val with the CV-chosen config; evaluate the
# held-out TEST set exactly once. Save the 9 models + metadata.
# ==========================================================================
import joblib

Xtv, Xte = fmat(trainval_df).values, fmat(test_df).values
Ytv, Yte = Ymat(trainval_df), Ymat(test_df)

final_models = fit_9(best_cfg, Xtv, to_train_space(Ytv))
pred_tv = predict_9(final_models, Xtv)
pred_te = predict_9(final_models, Xte)

payload = {"task": "regression", "target": TARGET, "configs": CONFIGS,
           "feature_cols": list(fmat(trainval_df).columns),
           "models": final_models, "config": best_cfg,
           "target_space": "log1p", "seed": RS}
joblib.dump(payload, f"artifacts/{ARTIFACT}")
print("saved artifacts/" + ARTIFACT, "| config:", best_cfg)

def per_config_table(Yt, Pt, Yv, Pv):
    rows = []
    for j, c in enumerate(CONFIGS):
        rows.append({"config": c,
                     "MAE_trainval": mean_absolute_error(Yv[:, j], Pv[:, j]),
                     "R2_trainval":  r2_score(Yv[:, j], Pv[:, j]),
                     "MAE_test":     mean_absolute_error(Yt[:, j], Pt[:, j]),
                     "R2_test":      r2_score(Yt[:, j], Pt[:, j])})
    tab = pd.DataFrame(rows)
    tab.loc[len(tab)] = {"config": "MEAN",
        "MAE_trainval": tab.MAE_trainval.mean(), "R2_trainval": tab.R2_trainval.mean(),
        "MAE_test": tab.MAE_test.mean(),         "R2_test": tab.R2_test.mean()}
    return tab

tab = per_config_table(Yte, pred_te, Ytv, pred_tv)
print("\n=== PER-CONFIG RESULTS (mean over the 9-vector at the bottom) ===")
print(tab.round(4).to_string(index=False))
print(f"\ngeneralization gap (test MAE - trainval MAE): "
      f"{tab.iloc[-1].MAE_test - tab.iloc[-1].MAE_trainval:.4f}  (small = good)")


## Inference — the length-9 output vector

In [ ]:
# ==========================================================================
# INFERENCE HELPER — one prompt in, a length-9 vector out (original units).
# The 9 entries line up with CONFIGS.
# ==========================================================================
def predict_vector(prompt_text, dataset):
    """dataset in {'gsm8k','arc_challenge','hellaswag'} — used only for the one-hot."""
    row = pd.DataFrame({"model_input": [str(prompt_text)], "dataset": [dataset]})
    X = fmat(row).values
    vec = predict_9(final_models, X)[0]
    return dict(zip(CONFIGS, vec))

# demo on the first held-out test prompt
_ex = test_df.iloc[0]
print("dataset:", _ex["dataset"], "| TARGET =", TARGET)
pred_vec = predict_vector(_ex["model_input"], _ex["dataset"])
for c in CONFIGS:
    print(f"  {c:14s} predicted={pred_vec[c]:.4g}   actual={_ex[c]:.4g}")


## How to read this

- The **CV table** is where hyperparameters are chosen — by mean validation score across all folds *and* repeats, with the std as an error bar. If two configs are within ~1 std, prefer the simpler (more regularized) one.
- The **per-config table** reports the held-out test result for each of the 9 configurations, with a **MEAN** row summarizing the length-9 output vector. The test set influenced neither the features nor the hyperparameters, so it is an honest estimate.
- `predict_vector(prompt, dataset)` shows the end use: one prompt in, the length-9 vector out.
- The saved `artifacts/*.joblib` bundles the 9 models, the feature-column order, the chosen config and the target space, so predictions can be reproduced elsewhere.